# 03 — Maintained weather transformations
Run after initial 01, or directly in the already initialized workspace.
This defines views against the maintained raw table and creates a provisional mapping only if absent.
Existing mapping choices are preserved on Run All. Deliberate remapping requires a separate reviewed edit.

Use `nyc_weather_hourly_with_precipitation` for analysis, especially its `precipitation_interpreted`,
`precipitation_interpretation`, `interpreted_precip_unknown`, trace and maintenance columns.
Older precipitation fields in the same view describe the original rule and may differ.
Views reflect daily raw updates automatically; this notebook does not need a daily schedule.

These are recorded-weather features, not prediction-ready availability-aware features.
The source-specific quality policy and same-time conflict selection remain provisional.
Full missing hours are absent; precipitation amounts are report accumulations, not clock-hour totals.

In [0]:
spark.conf.set("spark.sql.session.timeZone", "UTC")

## Observation view — always reads the maintained raw table

In [0]:
spark.sql("""
CREATE OR REPLACE VIEW workspace.default.noaa_weather_observations AS
SELECT
    STATION AS noaa_station_id,
    Station_name AS station_name,
    observed_at_utc,
    temperature,
    temperature_Quality_Code AS temperature_quality_code,
    temperature_Source_Code AS temperature_source_code,
    wind_speed,
    wind_speed_Quality_Code AS wind_quality_code,
    wind_speed_Source_Code AS wind_source_code,
    precipitation,
    precipitation_Measurement_Code AS precipitation_measurement_code,
    precipitation_Quality_Code AS precipitation_quality_code,
    precipitation_Source_Code AS precipitation_source_code,
    precipitation_Report_Type AS precipitation_report_type,
    CASE
        WHEN precipitation_Measurement_Code = 'T' THEN TRUE
        WHEN precipitation IS NOT NULL THEN FALSE
        ELSE NULL
    END AS precipitation_is_trace,
    CASE
        WHEN precipitation_Measurement_Code = 'T'
             OR precipitation > 0 THEN TRUE
        WHEN precipitation = 0 THEN FALSE
        ELSE NULL
    END AS precipitation_reported,
    source_url,
    snapshot_loaded_at_utc
FROM workspace.default.noaa_weather_raw
""")

## Hourly reported weather

In [0]:
spark.conf.set("spark.sql.session.timeZone", "UTC")

spark.sql("""
CREATE OR REPLACE VIEW workspace.default.nyc_weather_hourly AS

WITH hourly_reports AS (
    SELECT
        noaa_station_id,
        date_trunc('HOUR', observed_at_utc) AS hour_start_utc,
        COUNT(*) AS source_report_count,

        MAX(CASE WHEN temperature IS NOT NULL THEN named_struct(
            'observed_at', observed_at_utc,
            'source_code', temperature_source_code,
            'value', temperature,
            'quality_code', temperature_quality_code
        ) END) AS temp_report,

        MAX(CASE WHEN wind_speed IS NOT NULL THEN named_struct(
            'observed_at', observed_at_utc,
            'source_code', wind_source_code,
            'value', wind_speed,
            'quality_code', wind_quality_code
        ) END) AS wind_report,

        MAX(CASE
            WHEN precipitation IS NOT NULL
                 OR precipitation_is_trace = TRUE
            THEN named_struct(
                'observed_at', observed_at_utc,
                'source_code', precipitation_source_code,
                'value', precipitation,
                'quality_code', precipitation_quality_code,
                'measurement_code', precipitation_measurement_code,
                'report_type', precipitation_report_type,
                'is_trace', precipitation_is_trace
            )
        END) AS rain_report

    FROM workspace.default.noaa_weather_observations
    GROUP BY
        noaa_station_id,
        date_trunc('HOUR', observed_at_utc)
)

SELECT
    noaa_station_id,
    hour_start_utc,
    hour_start_utc + INTERVAL 1 HOUR AS hour_end_utc,
    source_report_count,

    temp_report.value AS temperature,
    temp_report.observed_at AS temperature_observed_at_utc,
    temp_report.source_code AS temperature_source_code,
    temp_report.quality_code AS temperature_quality_code,

    wind_report.value AS wind_speed,
    wind_report.observed_at AS wind_observed_at_utc,
    wind_report.source_code AS wind_source_code,
    wind_report.quality_code AS wind_quality_code,

    rain_report.value AS precipitation_latest_report,
    rain_report.observed_at AS precipitation_observed_at_utc,
    rain_report.source_code AS precipitation_source_code,
    rain_report.quality_code AS precipitation_quality_code,
    rain_report.measurement_code AS precipitation_measurement_code,
    rain_report.report_type AS precipitation_report_type,
    rain_report.is_trace AS precipitation_is_trace,

    temp_report IS NULL AS temperature_missing,
    wind_report IS NULL AS wind_missing,
    rain_report IS NULL AS precipitation_missing

FROM hourly_reports
""")

## Provisional mapping — preserve existing table; selection period is not a training split

In [0]:
spark.sql("""
CREATE TABLE IF NOT EXISTS
    workspace.default.citibike_noaa_mapping
USING DELTA
AS

WITH weather_coverage AS (
    SELECT
        noaa_station_id,
        COUNT(temperature) AS temperature_hours,
        COUNT(wind_speed) AS wind_hours
    FROM workspace.default.nyc_weather_hourly
    WHERE hour_start_utc >= TIMESTAMP '2026-08-21 00:00:00'
      AND hour_start_utc <  TIMESTAMP '2026-10-26 00:00:00'
    GROUP BY noaa_station_id
),

ranked AS (
    SELECT
        c.station_id,
        c.bike_station_name,
        c.bike_lat,
        c.bike_lon,
        c.snapshot_date AS bike_metadata_date,
        c.noaa_station_id,
        c.station_description,
        c.distance_km,
        w.temperature_hours,
        w.wind_hours,
        ROW_NUMBER() OVER (
            PARTITION BY c.station_id
            ORDER BY c.distance_km, c.noaa_station_id
        ) AS match_rank

    FROM workspace.default.citibike_noaa_candidates c
    JOIN weather_coverage w
        ON c.noaa_station_id = w.noaa_station_id
    WHERE w.temperature_hours > 0
      AND w.wind_hours > 0
)

SELECT
    *,
    'provisional_nearest_with_project_period_data' AS selection_rule,
    current_timestamp() AS mapping_created_at_utc
FROM ranked
WHERE match_rank = 1
""")

## Current six-station precipitation rules
Maintenance `$` is retained as a flag, not a blanket exclusion. NIL/PNO/PWINO exclusions remain.

In [0]:
spark.sql("""
CREATE OR REPLACE VIEW
workspace.default.noaa_precipitation_report_rules AS

WITH stations AS (
    SELECT * FROM VALUES
        ('USI0000KJRB', 'KJRB'),
        ('USW00014732', 'KLGA'),
        ('USW00014734', 'KEWR'),
        ('USW00094728', 'KNYC'),
        ('USW00094741', 'KTEB'),
        ('USW00094789', 'KJFK')
    AS stations(station_id, airport_code)
),

reports AS (
    SELECT
        r.*,
        s.airport_code,
        UPPER(TRIM(r.REM)) AS message
    FROM workspace.default.noaa_weather_raw r
    LEFT JOIN stations s ON r.STATION = s.station_id
),

matched AS (
    SELECT *,
        CASE WHEN
            REM_Source_Code = '413'
            AND airport_code IS NOT NULL
            AND (
                message LIKE CONCAT(
                    airport_code, ' ',
                    date_format(observed_at_utc, 'ddHHmm'), 'Z %'
                )
                OR message LIKE CONCAT(
                    'METAR ', airport_code, ' ',
                    date_format(observed_at_utc, 'ddHHmm'), 'Z %'
                )
                OR message LIKE CONCAT(
                    'SPECI ', airport_code, ' ',
                    date_format(observed_at_utc, 'ddHHmm'), 'Z %'
                )
            )
            AND message LIKE '% RMK %'
        THEN TRUE ELSE FALSE END AS matched_message
    FROM reports
),

rules AS (
    SELECT *,
        CASE WHEN
            matched_message
            AND message RLIKE '(^| )AO2( |$)'
            AND NOT message RLIKE '(^| )(NIL|PNO|PWINO)( |$)'
            AND NOT message RLIKE '(^| )P[0-9/]{4}( |$)'
            AND precipitation IS NULL
            AND COALESCE(precipitation_Measurement_Code, '') = ''
        THEN TRUE ELSE FALSE END AS inferred_zero
    FROM matched
)

SELECT
    STATION AS noaa_station_id,
    observed_at_utc,
    precipitation AS precipitation_original,
    CASE
        WHEN inferred_zero THEN CAST(0 AS DOUBLE)
        ELSE precipitation
    END AS precipitation_interpreted,

    CASE
        WHEN precipitation_Measurement_Code = 'T' THEN 'trace_reported'
        WHEN precipitation IS NOT NULL THEN 'numeric_reported'
        WHEN inferred_zero THEN 'inferred_zero_asos'
        ELSE 'unknown'
    END AS interpretation,

    CASE
        WHEN precipitation_Measurement_Code = 'T' THEN TRUE
        WHEN precipitation IS NOT NULL OR inferred_zero THEN FALSE
        ELSE NULL
    END AS is_trace,

    precipitation_Quality_Code AS quality_code,
    precipitation_Source_Code AS precipitation_source_code,
    REM_Source_Code AS message_source_code,
    message AS original_message,
    CASE
        WHEN message IS NULL THEN NULL
        ELSE INSTR(message, '$') > 0
    END AS maintenance_flag,

    -- Include complete matched messages even when precipitation is unknown.
    -- This prevents an earlier wet report from hiding a later unavailable report.
    (
        matched_message
        OR precipitation IS NOT NULL
        OR COALESCE(precipitation_Measurement_Code = 'T', FALSE)
    ) AS use_for_hourly_selection

FROM rules
""")

## Revised hourly precipitation
Keep maintenance last in the comparison struct to preserve the existing deterministic tie-break.

In [0]:
spark.sql("""
CREATE OR REPLACE VIEW
workspace.default.nyc_weather_hourly_with_precipitation AS

WITH hourly_precip AS (
    SELECT
        noaa_station_id,
        date_trunc('HOUR', observed_at_utc) AS hour_start_utc,
        MAX(named_struct(
            'observed_at', observed_at_utc,
            'interpretation', interpretation,
            'value', precipitation_interpreted,
            'original_value', precipitation_original,
            'is_trace', is_trace,
            'quality_code', quality_code,
            'original_message', original_message,
            'maintenance_flag', maintenance_flag
        )) AS report
    FROM workspace.default.noaa_precipitation_report_rules
    WHERE use_for_hourly_selection
    GROUP BY
        noaa_station_id,
        date_trunc('HOUR', observed_at_utc)
)

SELECT
    h.*,
    p.report.value AS precipitation_interpreted,
    p.report.observed_at AS interpreted_precip_observed_at_utc,
    COALESCE(p.report.interpretation, 'unknown')
        AS precipitation_interpretation,
    p.report.is_trace AS interpreted_precip_is_trace,
    p.report.quality_code AS interpreted_precip_quality_code,
    p.report.maintenance_flag AS precipitation_maintenance_flag,
    COALESCE(p.report.interpretation, 'unknown') = 'unknown'
        AS interpreted_precip_unknown
FROM workspace.default.nyc_weather_hourly h
LEFT JOIN hourly_precip p
    ON h.noaa_station_id = p.noaa_station_id
   AND h.hour_start_utc = p.hour_start_utc
""")

display(spark.sql("""
SELECT
    noaa_station_id,
    precipitation_interpretation,
    COUNT(*) AS hours
FROM workspace.default.nyc_weather_hourly_with_precipitation
WHERE hour_start_utc >= TIMESTAMP '2026-08-21 00:00:00'
  AND hour_start_utc <  TIMESTAMP '2026-10-26 00:00:00'
GROUP BY noaa_station_id, precipitation_interpretation
ORDER BY noaa_station_id, precipitation_interpretation
"""))

## Freshness monitoring view

In [0]:
spark.sql("""
CREATE OR REPLACE VIEW workspace.default.noaa_weather_freshness AS
WITH latest AS (
    SELECT
        noaa_station_id,
        MAX(observed_at_utc) AS latest_observation_utc,
        MAX(snapshot_loaded_at_utc) AS latest_refresh_utc
    FROM workspace.default.noaa_weather_observations
    GROUP BY noaa_station_id
),
mapped AS (
    SELECT
        noaa_station_id,
        COUNT(*) AS mapped_bike_stations
    FROM workspace.default.citibike_noaa_mapping
    GROUP BY noaa_station_id
)
SELECT
    r.noaa_station_id,
    r.station_name,
    COALESCE(m.mapped_bike_stations, 0) AS mapped_bike_stations,
    l.latest_observation_utc,
    l.latest_refresh_utc,
    ROUND(
        (unix_timestamp(current_timestamp())
         - unix_timestamp(l.latest_observation_utc)) / 3600.0,
        1
    ) AS observation_age_hours,
    ROUND(
        (unix_timestamp(current_timestamp())
         - unix_timestamp(l.latest_refresh_utc)) / 3600.0,
        1
    ) AS refresh_age_hours
FROM workspace.default.noaa_station_registry r
LEFT JOIN latest l USING (noaa_station_id)
LEFT JOIN mapped m USING (noaa_station_id)
""")

display(spark.table("workspace.default.noaa_weather_freshness"))

## Catalog documentation

In [0]:
spark.sql("""
COMMENT ON TABLE workspace.default.citibike_noaa_mapping IS
'Provisional mapping of Citi Bike stations to nearby NOAA GHCNh stations.
Selected by distance among downloaded stations with temperature and wind
records during August 21–October 25, 2026. Uses available station metadata;
historical location changes are not resolved. Selection must be reviewed
using training-period data before model evaluation.'
""")

spark.sql("""
COMMENT ON TABLE
workspace.default.nyc_weather_hourly_with_precipitation IS
'Hourly view of NOAA GHCNh observations. Temperature and wind use the
latest non-null report within each UTC hour. Precipitation_interpreted
contains the selected report accumulation, not a clock-hour total.
Inferred dry reports are labeled inferred_zero_asos; traces, unknowns,
original quality codes, and maintenance flags are retained.
Entirely absent hours have no row. Quality screening is provisional.
Observation time is not publication time: account for archive delay
before using these data as prediction features.'
""")

Validation examples are in 04. For existing installations, test cleaned 02 manually as a job before changing its schedule.